# ADA 4 CPU branch and expert dispatch

The primary snapshot is `Ada_4/ada_v4.py` (SHA-256 `3df9256eafc3ec22b0bf9de9d425d841fc503c1da04c912894870c7929c2d8ad`).
`Ada_4/ada_v4_production.py` is a different variant and is not executed.
Internal project drafts informed nomenclature. The runnable fixture covers
the sequential Mamba-style scan, source MoD, manual causal GQA, shared and
routed experts, and weighted multi-token prediction.

The notebook extracts arithmetic from the CPU branches and reconnects
those components under a small config. GPU-only dispatch, HF wrappers,
diffusion, retrieval, telemetry and distributed wrappers are outside the
executed scope. The source GPU-labeled branches require separate
implementation and parity evidence.

## Run this notebook

Use Python 3.11 or later, JupyterLab and a CPU PyTorch build. In an isolated virtual
environment, install `jupyterlab`, `ipykernel` and `torch>=2.6`, then open this
notebook and run all cells. The reference run is deterministic on CPU and neither
reads repository data nor downloads models. Training starts from random weights.

In [1]:
import json
import math
from dataclasses import asdict, dataclass, field, replace
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

SEED = 2031
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}, device={DEVICE}, seed={SEED}")

import warnings

PyTorch 2.11.0+cpu, device=cpu, seed=2031


## CPU architecture

The source scan applies a softplus timestep and discretizes the input
projection before recurrent updates. The attention uses four query heads
sharing two key/value heads. A separate shared expert is always computed.
The v4 source chooses selected-token expert dispatch during training and
dense output weighting during evaluation. Neither path uses a fused kernel
in this notebook.

In [2]:
@dataclass(frozen=True)
class ADAConfig:
    vocab_size: int = 23
    d_model: int = 32
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 2
    mamba_d_state: int = 4
    mamba_d_conv: int = 4
    mamba_expand: int = 2
    mamba_n_heads: int = 4
    mamba_dt_min: float = 0.001
    mamba_dt_max: float = 0.1
    mamba_conv_bias: bool = True
    n_experts: int = 4
    n_active_experts: int = 2
    n_shared_experts: int = 1
    moe_d_ff: int = 24
    ffn_dropout: float = 0.0
    moe_router_jitter_noise: float = 0.0
    moe_bias_lr: float = 0.001
    moe_grouped_gemm: bool = True
    mod_capacity: float = 1.0
    n_mtp: int = 2
    mtp_loss_weights: tuple[float, float] = (1.0, 0.8)
    mtp_share_trunk: bool = True
    max_seq_len: int = 32
    rope_theta: float = 500_000.0
    rope_scaling: Optional[Dict[str, Any]] = None
    norm_eps: float = 1e-6
    dropout: float = 0.0

    @property
    def head_dim(self):
        return self.d_model // self.n_heads

    @property
    def mamba_inner(self):
        return self.mamba_expand * self.d_model

    @property
    def mamba_head_dim(self):
        return self.mamba_inner // self.mamba_n_heads

    @property
    def block_pattern(self):
        base = ["mamba", "mamba", "mamba", "moe_attn"]
        return [base[index % len(base)] for index in range(self.n_layers)]

In [3]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (no bias)."""

    def __init__(self, dim: int, eps: float = 1e-6, use_compiled: bool = False):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        fp32 = x.float()
        norm = fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (fp32 * norm).to(x.dtype) * self.weight


class SwiGLU(nn.Module):
    """SwiGLU FFN: W_down(SiLU(W_gate(x)) * W_up(x))"""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0, bias: bool = False):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=bias)
        self.w_up = nn.Linear(d_model, d_ff, bias=bias)
        self.w_down = nn.Linear(d_ff, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.w_down(self.dropout(F.silu(self.w_gate(x)) * self.w_up(x)))


class RoPECache(nn.Module):
    """Precomputed RoPE cache with YaRN scaling support."""

    def __init__(self, max_seq_len: int, head_dim: int, theta: float = 500_000.0,
                 scaling: Optional[Dict[str, Any]] = None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.head_dim = head_dim
        self.theta = theta
        self.scaling = scaling

        cos, sin = self._build_cache(max_seq_len, head_dim, theta, scaling)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    def _build_cache(self, seq_len: int, head_dim: int, theta: float,
                     scaling: Optional[Dict[str, Any]]) -> Tuple[Tensor, Tensor]:
        half = head_dim // 2
        inv_freq = 1.0 / (theta ** (torch.arange(0, half, dtype=torch.float32) / half))

        if scaling is not None:
            scale_type = scaling.get("type", "linear")
            factor = scaling.get("factor", 1.0)
            if scale_type == "linear":
                inv_freq = inv_freq / factor
            elif scale_type == "yarn":
                # YaRN scaling
                beta_fast = scaling.get("beta_fast", 32)
                beta_slow = scaling.get("beta_slow", 1)
                original_max_seq = scaling.get("original_max_seq_len", 4096)
                # Simplified YaRN - full impl in HF transformers
                inv_freq = inv_freq / factor
            else:
                warnings.warn(f"Unknown RoPE scaling type: {scale_type}")

        t = torch.arange(seq_len, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        return freqs.cos(), freqs.sin()

    def forward(self, x: Tensor, position_ids: Optional[Tensor] = None) -> Tensor:
        """Apply RoPE to x (B, H, T, D)."""
        return apply_rope(x, self.cos, self.sin, position_ids)


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor,
               position_ids: Optional[Tensor] = None) -> Tensor:
    """Apply rotary position embeddings."""
    T = x.shape[2]
    half = x.shape[-1] // 2

    if position_ids is not None:
        c = cos[position_ids].unsqueeze(1)  # (B, 1, T, half)
        s = sin[position_ids].unsqueeze(1)
    else:
        c = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, half)
        s = sin[:T].unsqueeze(0).unsqueeze(0)

    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1)

In [4]:
class Mamba2SSM(nn.Module):
    """
    Mamba-2 Structured State Space Duality (SSD) Layer.
    Notebook adapter retains the source's sequential CPU branch.
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        di = cfg.mamba_inner
        N = cfg.mamba_d_state
        H = cfg.mamba_n_heads
        hd = cfg.mamba_head_dim

        self.n_heads = H
        self.head_dim = hd
        self.d_state = N
        self.d_inner = di

        # Fused input projection: z + x + B + C + dt
        self.in_proj = nn.Linear(cfg.d_model, 2 * di + 2 * N + H, bias=False)

        # Causal depthwise conv
        self.conv1d = nn.Conv1d(
            di, di,
            kernel_size=cfg.mamba_d_conv,
            groups=di,
            padding=cfg.mamba_d_conv - 1,
            bias=cfg.mamba_conv_bias,
        )

        # dt bias (log-uniform init)
        log_dt = (
            torch.rand(H) * (math.log(cfg.mamba_dt_max) - math.log(cfg.mamba_dt_min))
            + math.log(cfg.mamba_dt_min)
        )
        self.dt_bias = nn.Parameter(log_dt)

        # A_log: A[h] = -exp(A_log[h]) < 0
        self.A_log = nn.Parameter(torch.log(torch.arange(1, H + 1, dtype=torch.float32)))

        # D: per-head skip
        self.D = nn.Parameter(torch.ones(H))

        # Output norm + projection
        self.out_norm = RMSNorm(di, cfg.norm_eps)
        self.out_proj = nn.Linear(di, cfg.d_model, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        B, L, _ = x.shape
        H, hd, N, di = self.n_heads, self.head_dim, self.d_state, self.d_inner

        # Fused projection
        proj = self.in_proj(x)
        z, x_in, B_in, C_in, dt = proj.split([di, di, N, N, H], dim=-1)

        # Causal conv1d
        x_in = F.silu(self.conv1d(x_in.transpose(1, 2))[..., :L].transpose(1, 2))

        # Reshape to per-head
        x_h = x_in.reshape(B, L, H, hd)
        z_h = z.reshape(B, L, H, hd)

        # Discretization
        A = -torch.exp(self.A_log.float())  # (H,)
        dt = F.softplus(dt.float() + self.dt_bias)  # (B, L, H)
        Abar = torch.exp(dt * A)  # (B, L, H)
        Bbar = dt.unsqueeze(-1) * B_in.float().unsqueeze(2)  # (B, L, H, N)

        y = self._sequential_scan(x_h, Abar, Bbar, C_in, z_h)

        # Gate + project
        y = self.out_norm(y.reshape(B, L, di))
        y = y * F.silu(z)
        return self.out_proj(y)

    def _sequential_scan(self, x_h: Tensor, Abar: Tensor, Bbar: Tensor,
                         C_in: Tensor, z_h: Tensor) -> Tensor:
        """Sequential Python scan (fallback)."""
        B, L, H, hd = x_h.shape
        N = self.d_state
        h = x_h.new_zeros(B, H, hd, N, dtype=torch.float32)
        ys = x_h.new_empty(B, L, H, hd)

        for t in range(L):
            a_t = Abar[:, t, :, None, None]
            b_t = Bbar[:, t, :, None, :]
            u_t = x_h[:, t, :, :, None].float()

            h = a_t * h + b_t * u_t

            c_t = C_in[:, t, None, None, :].float()
            y_skip = self.D[None, :, None] * x_h[:, t].float()
            ys[:, t] = ((h * c_t).sum(-1) + y_skip).to(x_h.dtype)

        return ys



class Mamba2Block(nn.Module):
    """Pre-norm Mamba-2 block (residual applied by MoD wrapper)."""

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ssm = Mamba2SSM(cfg)

    def forward(self, x: Tensor) -> Tensor:
        return self.ssm(self.norm(x))


class MoD(nn.Module):
    """
    Mixture of Depths wrapper.
    Selects top-(capacity * T) tokens by learned router score.
    Selected tokens: x_out = x + sigmoid(router_score) * block(x)
    Unselected tokens: x_out = x (identity)
    """

    def __init__(self, block: nn.Module, d_model: int, capacity: float,
                 apply_to_attn: bool = False):
        super().__init__()
        self.block = block
        self.capacity = capacity
        self.router = nn.Linear(d_model, 1, bias=False)
        self.apply_to_attn = apply_to_attn

    def forward(self, x: Tensor, position_ids: Optional[Tensor] = None) -> Tensor:
        B, T, D = x.shape
        k = max(1, int(T * self.capacity))

        scores = self.router(x).squeeze(-1)  # (B, T)
        _, top_idx = torch.topk(scores, k, dim=-1, sorted=False)

        # Router weight: sigmoid of score at selected positions
        weights = torch.sigmoid(scores.gather(1, top_idx))  # (B, k)

        # Sort by position index (crucial for causal correctness)
        top_idx_sorted, perm = top_idx.sort(dim=-1)
        weights_sorted = weights.gather(1, perm)

        # Gather selected tokens
        idx_exp = top_idx_sorted.unsqueeze(-1).expand(B, k, D)
        x_sel = x.gather(1, idx_exp)  # (B, k, D)

        # Process selected tokens
        if self.apply_to_attn and hasattr(self.block, 'forward'):
            # For attention blocks, need position_ids
            pos_sel = position_ids.gather(1, top_idx_sorted) if position_ids is not None else None
            block_out = self.block(x_sel, pos_sel)
        else:
            block_out = self.block(x_sel)

        # Weighted residual
        x_proc = x_sel + weights_sorted.unsqueeze(-1) * block_out

        # Scatter back
        out = x.clone()
        out.scatter_(1, idx_exp, x_proc)
        return out

In [5]:
class Expert(nn.Module):
    """Single SwiGLU expert."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.w_down(self.dropout(F.silu(self.w_gate(x)) * self.w_up(x)))


class SharedExpert(nn.Module):
    """Shared expert (always active) - larger capacity."""

    def __init__(self, d_model: int, d_ff: int, n_shared: int = 4, dropout: float = 0.0):
        super().__init__()
        # Shared experts have larger FFN (typically 2-4x)
        shared_d_ff = d_ff * n_shared
        self.w_gate = nn.Linear(d_model, shared_d_ff, bias=False)
        self.w_up = nn.Linear(d_model, shared_d_ff, bias=False)
        self.w_down = nn.Linear(shared_d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.w_down(self.dropout(F.silu(self.w_gate(x)) * self.w_up(x)))


class DeepSeekV3MoE(nn.Module):
    """
    DeepSeek-V3 Fine-Grained MoE:
    - 256 fine-grained experts (d_ff=2048 each)
    - Top-4 routing
    - 4 shared experts (always active)
    - Bias-corrected load balancing (NO auxiliary loss)
    - Python selected-token dispatch, with a separate dense reference
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.n_experts = cfg.n_experts
        self.n_active = cfg.n_active_experts
        self.n_shared = cfg.n_shared_experts
        self.bias_lr = cfg.moe_bias_lr
        self.jitter_noise = cfg.moe_router_jitter_noise
        self.grouped_gemm = cfg.moe_grouped_gemm

        # Router
        self.router = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)

        # Fine-grained experts
        self.experts = nn.ModuleList([
            Expert(cfg.d_model, cfg.moe_d_ff, cfg.ffn_dropout)
            for _ in range(cfg.n_experts)
        ])

        # Shared experts (always active)
        self.shared_experts = SharedExpert(
            cfg.d_model, cfg.moe_d_ff, cfg.n_shared_experts, cfg.ffn_dropout
        )

        # Load balancing bias (buffer, not parameter - no gradients)
        self.register_buffer("expert_bias", torch.zeros(cfg.n_experts))
        self.register_buffer("expert_load", torch.zeros(cfg.n_experts))

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor]:
        B, T, D = x.shape
        n_tok = B * T
        x_flat = x.reshape(n_tok, D)

        # Router logits
        logits = self.router(x_flat)  # (n_tok, E)

        # Jitter noise for exploration (training only)
        if self.training and self.jitter_noise > 0:
            logits = logits + torch.randn_like(logits) * self.jitter_noise

        # Bias-corrected routing
        biased_logits = logits + self.expert_bias
        _, topk_idx = torch.topk(biased_logits, self.n_active, dim=-1)  # (n_tok, k)

        # Compute weights from UNBIASED logits (gradient flows here)
        topk_logits = logits.gather(1, topk_idx)
        weights = F.softmax(topk_logits.float(), dim=-1).to(x.dtype)  # (n_tok, k)

        # Shared expert output (always computed)
        shared_out = self.shared_experts(x_flat)  # (n_tok, D)

        # Dispatch to fine-grained experts
        if self.grouped_gemm and self.training:
            expert_out = self._grouped_gemm_dispatch(x_flat, topk_idx, weights)
        else:
            expert_out = self._dense_dispatch(x_flat, topk_idx, weights)

        # Combine: shared + routed
        out = shared_out + expert_out

        # Update bias (training only, no grad)
        if self.training:
            self._update_bias(topk_idx, n_tok)

        return out.reshape(B, T, D), self.expert_load.clone()

    def _dense_dispatch(self, x_flat: Tensor, topk_idx: Tensor, weights: Tensor) -> Tensor:
        """Dense dispatch: compute all experts, combine with weights."""
        n_tok = x_flat.shape[0]
        E = self.n_experts

        # Stack all expert outputs: (n_tok, E, D)
        expert_outs = torch.stack([exp(x_flat) for exp in self.experts], dim=1)

        # Build combine matrix
        combine = torch.zeros(n_tok, E, device=x_flat.device, dtype=x_flat.dtype)
        combine.scatter_(1, topk_idx, weights)

        # Weighted sum
        return torch.einsum("te,ted->td", combine, expert_outs)

    def _grouped_gemm_dispatch(self, x_flat: Tensor, topk_idx: Tensor, weights: Tensor) -> Tensor:
        """
        Python selected-token dispatch; no fused grouped matrix kernel.
        Only computes experts that have tokens assigned.
        """
        n_tok = x_flat.shape[0]
        D = x_flat.shape[-1]
        k = self.n_active

        # Count tokens per expert
        expert_counts = torch.zeros(self.n_experts, device=x_flat.device, dtype=torch.long)
        expert_counts.scatter_add_(0, topk_idx.flatten(),
                                   torch.ones(n_tok * k, device=x_flat.device, dtype=torch.long))

        # For each expert with tokens, compute
        out = torch.zeros_like(x_flat)
        for e in range(self.n_experts):
            count = expert_counts[e].item()
            if count == 0:
                continue

            # Find tokens routed to this expert
            token_mask = (topk_idx == e).any(dim=-1)  # (n_tok,)
            if not token_mask.any():
                continue

            token_indices = token_mask.nonzero(as_tuple=True)[0]
            expert_x = x_flat[token_indices]  # (count, D)

            # Compute expert output
            expert_out = self.experts[e](expert_x)  # (count, D)

            # Get weights for this expert
            # topk_idx[token_indices] has shape (count, k) - find which position
            expert_pos = (topk_idx[token_indices] == e).nonzero(as_tuple=True)[1]
            expert_weights = weights[token_indices, expert_pos]  # (count,)

            out[token_indices] += expert_out * expert_weights.unsqueeze(-1)

        return out

    def _update_bias(self, topk_idx: Tensor, n_tok: int):
        """Bias-corrected load balancing (DeepSeek-V3 §2.3)."""
        with torch.no_grad():
            # Count tokens per expert
            load = torch.zeros(self.n_experts, device=topk_idx.device, dtype=torch.float32)
            load.scatter_add_(0, topk_idx.flatten(),
                              torch.ones_like(topk_idx.flatten(), dtype=torch.float32))

            # Normalize: tokens_per_expert / (n_tok * k)
            load_frac = load / (n_tok * self.n_active)

            # Update bias: bias += lr * (1/E - load_frac)
            self.expert_bias.add_(self.bias_lr * (1.0 / self.n_experts - load_frac))

            # Track load for monitoring
            self.expert_load.copy_(load_frac)

In [6]:
class GQAAttention(nn.Module):
    """
    Grouped Query Attention with RoPE.
    Notebook adapter retains the source's manual causal path.
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.n_rep = cfg.n_heads // cfg.n_kv_heads
        self.scale = self.head_dim ** -0.5

        D, H, Hkv, hd = cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.wq = nn.Linear(D, H * hd, bias=False)
        self.wk = nn.Linear(D, Hkv * hd, bias=False)
        self.wv = nn.Linear(D, Hkv * hd, bias=False)
        self.wo = nn.Linear(H * hd, D, bias=False)
        self.dropout = nn.Dropout(cfg.dropout) if cfg.dropout > 0 else nn.Identity()

        # RoPE cache
        self.rope = RoPECache(cfg.max_seq_len, cfg.head_dim, cfg.rope_theta, cfg.rope_scaling)

    def forward(self, x: Tensor, position_ids: Optional[Tensor] = None,
                attn_mask: Optional[Tensor] = None) -> Tensor:
        B, T, _ = x.shape

        q = self.wq(x).reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).reshape(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.wv(x).reshape(B, T, self.n_kv, self.head_dim).transpose(1, 2)

        # Apply RoPE
        q = self.rope(q, position_ids)
        k = self.rope(k, position_ids)

        # GQA: repeat KV heads
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        out = self._manual_attn(q, k, v, attn_mask)

        out = out.transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)

    def _manual_attn(self, q: Tensor, k: Tensor, v: Tensor, mask: Optional[Tensor]) -> Tensor:
        """Manual attention fallback."""
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        else:
            causal = torch.tril(torch.ones(q.shape[2], q.shape[2], device=q.device, dtype=torch.bool))
            scores = scores.masked_fill(~causal[None, None], float("-inf"))
        attn = F.softmax(scores.float(), dim=-1).to(q.dtype)
        attn = self.dropout(attn)
        return torch.matmul(attn, v)

    def _make_causal_mask(self, T: int, device: torch.device) -> Tensor:
        return torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))


class MoEAttnBlock(nn.Module):
    """Pre-norm GQA + Pre-norm DeepSeek-V3 MoE."""

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.moe = DeepSeekV3MoE(cfg)
        self.dropout = nn.Dropout(cfg.dropout) if cfg.dropout > 0 else nn.Identity()

    def forward(self, x: Tensor, position_ids: Optional[Tensor] = None) -> Tuple[Tensor, Tensor]:
        # Attention
        x = x + self.dropout(self.attn(self.attn_norm(x), position_ids))
        # MoE FFN
        moe_out, load = self.moe(self.ffn_norm(x))
        x = x + self.dropout(moe_out)
        return x, load

In [7]:
class MTPHead(nn.Module):
    """Single MTP head: predicts token at t + offset."""

    def __init__(self, d_model: int, vocab_size: int, use_proj: bool = True):
        super().__init__()
        self.use_proj = use_proj
        if use_proj:
            self.proj = nn.Linear(d_model, d_model, bias=False)
            self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        if self.use_proj:
            x = self.norm(self.proj(x))
        return self.head(x)


class MTPModule(nn.Module):
    """
    Multi-Token Prediction v2 (Gloeckle et al. 2024).
    - n_mtp heads predicting t+2 ... t+n_mtp+1
    - Shared trunk, light projection heads
    - Loss weights: 1.0, 0.8, 0.6, 0.4, 0.2, 0.1, 0.05, 0.025
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg
        self.heads = nn.ModuleList([
            MTPHead(cfg.d_model, cfg.vocab_size, cfg.mtp_share_trunk)
            for _ in range(cfg.n_mtp)
        ])
        self.loss_weights = cfg.mtp_loss_weights

    def forward(self, x: Tensor) -> List[Tensor]:
        return [head(x) for head in self.heads]

    def compute_loss(self, mtp_logits: List[Tensor], targets: Tensor) -> Tensor:
        """Compute weighted MTP loss."""
        total_loss = 0.0
        V = self.cfg.vocab_size
        for i, logits in enumerate(mtp_logits):
            offset = i + 2
            T = targets.shape[1]
            if T - offset <= 0:
                continue
            loss = F.cross_entropy(
                logits[:, :T - offset].reshape(-1, V),
                targets[:, offset:].reshape(-1),
                reduction='mean'
            )
            total_loss += self.loss_weights[i] * loss
        return total_loss

In [8]:
class ADA4(nn.Module):
    # The v4 CPU AR path with optional runtime branches removed.

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList(
            MoD(Mamba2Block(cfg), cfg.d_model, cfg.mod_capacity)
            if kind == "mamba" else MoEAttnBlock(cfg)
            for kind in cfg.block_pattern
        )
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        self.mtp = MTPModule(cfg)
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Conv1d)):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, ids):
        x = self.embed(ids)
        loads = []
        for layer in self.layers:
            if isinstance(layer, MoEAttnBlock):
                x, load = layer(x)
                loads.append(load)
            else:
                x = layer(x)
        hidden = self.norm(x)
        return self.lm_head(hidden), self.mtp(hidden), loads

    def compute_loss(self, ids):
        logits, auxiliary, _ = self(ids)
        main = F.cross_entropy(logits[:, :-1].flatten(0, 1), ids[:, 1:].flatten())
        mtp = self.mtp.compute_loss(auxiliary, ids)
        return main + 0.3 * mtp, {"next_token_ce": float(main.detach()),
                                  "weighted_mtp_ce": float(mtp.detach())}


cfg = ADAConfig()
build_model = ADA4

## Data and optimization

A synthetic vocabulary of size 23 represents fixed-step arithmetic sequences.
Evaluation uses an independent RNG stream. The objective combines next-token
cross-entropy with weighted predictions two and three steps ahead. Capacity
one isolates the AR backbone from the historical global MoD budget.

In [9]:
def arithmetic_sequences(seed, count, length=12, vocab_size=23):
    rng = torch.Generator().manual_seed(seed)
    start = torch.randint(0, vocab_size, (count, 1), generator=rng)
    step = torch.randint(1, 5, (count, 1), generator=rng)
    return (start + step * torch.arange(length)) % vocab_size


train_rows = arithmetic_sequences(SEED + 1, 16)
eval_rows = arithmetic_sequences(SEED + 2, 4)
assert train_rows.dtype == torch.long and eval_rows.shape == (4, 12)
print("Synthetic train/eval shapes:", tuple(train_rows.shape), tuple(eval_rows.shape))

Synthetic train/eval shapes: (16, 12) (4, 12)


In [10]:
assert cfg.block_pattern == ["mamba", "mamba", "mamba", "moe_attn"]
model = build_model(cfg).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=0.002)


@torch.inference_mode()
def evaluate(ids):
    model.eval()
    logits = model(ids)[0][:, :-1]
    next_ids = ids[:, 1:]
    ce = F.cross_entropy(logits.flatten(0, 1), next_ids.flatten())
    accuracy = (logits.argmax(-1) == next_ids).float().mean()
    return float(ce), float(accuracy)


before, _ = evaluate(eval_rows)
model.train()
metrics = None
for step in range(8):
    batch = train_rows[(step % 4) * 4 : (step % 4 + 1) * 4]
    opt.zero_grad(set_to_none=True)
    loss, metrics = model.compute_loss(batch)
    loss.backward()
    norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(loss) and torch.isfinite(norm) and norm > 0
    opt.step()
after, accuracy = evaluate(eval_rows)
assert math.isfinite(after) and 0.0 <= accuracy <= 1.0

with torch.inference_mode():
    ids = eval_rows[:1, :4].clone()
    for _ in range(3):
        logits, auxiliary, loads = model(ids)
        assert logits.shape == (*ids.shape, cfg.vocab_size)
        assert len(auxiliary) == cfg.n_mtp
        assert len(loads) == 1 and torch.isfinite(loads[0]).all()
        ids = torch.cat((ids, logits[:, -1].argmax(dim=-1, keepdim=True)), dim=1)
    assert ids.shape == (1, 7)

print(json.dumps({
    "fixture": "fixed-step modulo arithmetic",
    "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    "validation_targets": eval_rows[:, 1:].numel(),
    "before_ce": round(before, 4),
    "after_ce": round(after, 4),
    "accuracy": round(accuracy, 4),
    "train_last": {name: round(value, 4) for name, value in metrics.items()},
    "sample_ids": ids[0].tolist(),
}, indent=2))

{
  "fixture": "fixed-step modulo arithmetic",
  "parameter_count": 40100,
  "validation_targets": 44,
  "before_ce": 3.1669,
  "after_ce": 2.8148,
  "accuracy": 0.2273,
  "train_last": {
    "next_token_ce": 2.7006,
    "weighted_mtp_ce": 4.9888
  },
  "sample_ids": [
    19,
    22,
    2,
    5,
    5,
    9,
    13
  ]
}


## Correctness gates

Prefix equality must hold for the reference mixer and full tiny decoder
with all tokens routed. Separately reproduce the original top-k budget's
prefix failure. Use fixed expert assignments to compare selected-token
dispatch to the dense reference and verify that empty experts never run.

In [11]:
original = eval_rows[:2].clone()
altered_suffix = original.clone()
cut = 6
altered_suffix[:, cut:] = (altered_suffix[:, cut:] + 10) % cfg.vocab_size
with torch.inference_mode():
    full, aux, _ = model(original)
    altered = model(altered_suffix)[0]
    truncated = model(original[:, :cut])[0]
    input_features = model.embed(original)
    ssm = model.layers[0].block
    ssm_full = ssm(input_features)
    ssm_prefix = ssm(input_features[:, :cut])
    torch.testing.assert_close(full[:, :cut], altered[:, :cut], rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(full[:, :cut], truncated, rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(ssm_full[:, :cut], ssm_prefix, rtol=1e-5, atol=1e-5)
    assert all(logits.shape == full.shape for logits in aux)
print("PASS: SSM and full decoder prefix consistency in the all-token routing control.")


def mod_ranking_counterexample():
    # Weight 1 on feature 0: token 1 wins the 50% budget at length 2,
    # but appending a higher score at length 3 displaces it.
    class FixedContribution(nn.Module):
        def forward(self, inputs):
            return torch.ones_like(inputs)

    test_router = MoD(FixedContribution(), 1, 0.5).eval()
    with torch.inference_mode():
        test_router.router.weight.fill_(1)
        if test_router.router.bias is not None:
            test_router.router.bias.zero_()
        inputs = torch.tensor([[[0.0], [1.0], [10.0]]])
        prefix = test_router(inputs[:, :2])
        full = test_router(inputs)
        return float((prefix - full[:, :2]).abs().max())


delta = mod_ranking_counterexample()
assert delta > 1e-3
print(f"EXPECTED FAILURE: historical 50% global top-k MoD is not prefix invariant (delta={delta:.6f}).")

PASS: SSM and full decoder prefix consistency in the all-token routing control.
EXPECTED FAILURE: historical 50% global top-k MoD is not prefix invariant (delta=0.731059).


In [12]:
moe = model.layers[-1].moe
split = torch.linspace(-1.0, 1.0, steps=5 * cfg.d_model).reshape(5, cfg.d_model)
fixed_indices = torch.tensor([[0, 1], [1, 0], [0, 1], [1, 0], [0, 1]])
fixed_weights = torch.tensor([[0.75, 0.25]] * 5)

with torch.inference_mode():
    dense = moe._dense_dispatch(split, fixed_indices, fixed_weights)
    counts = [0 for _ in moe.experts]
    hooks = [
        expert.register_forward_hook(
            lambda _expert, args, _output, i=index: counts.__setitem__(i, args[0].shape[0])
        )
        for index, expert in enumerate(moe.experts)
    ]
    try:
        sparse = moe._grouped_gemm_dispatch(split, fixed_indices, fixed_weights)
    finally:
        for hook in hooks:
            hook.remove()
    torch.testing.assert_close(dense, sparse, atol=1e-6, rtol=1e-5)
    assert counts == [5, 5, 0, 0]

    def shifted_ce(logits, ids, shift):
        return F.cross_entropy(logits[:, :-shift].flatten(0, 1), ids[:, shift:].flatten())

    _, horizons, _ = model(original)
    observed_mtp = model.mtp.compute_loss(horizons, original)
    expected_mtp = sum(weight * shifted_ce(logits, original, horizon)
                       for horizon, (weight, logits) in
                       enumerate(zip(cfg.mtp_loss_weights, horizons), 2))
    torch.testing.assert_close(observed_mtp, expected_mtp)
print(f"PASS: selected-token dispatch equals dense reference, processed rows {counts}, "
      "and weighted MTP targets align at offsets 2 and 3.")

PASS: selected-token dispatch equals dense reference, processed rows [5, 5, 0, 0], and weighted MTP targets align at offsets 2 and 3.


## Interpretation

The executed differences from v3 include always-active shared experts,
selected-token dispatch in training, and a weighted projected MTP loss.
The result is CPU reference evidence. It makes no GPU speed, kernel parity,
distributed training, language benchmark, or speculative decoding claim.